In [4]:
import os
from pathlib import Path

# ============== Your training variables ==============
os.environ["RUN_NAME_FULL"] = "qwen2_5_vl_lora_aggplus"
os.environ["OUT_DIR_FULL"] = str(Path(f"~/llm_spine_parse_train_public/training_output/{os.environ['RUN_NAME_FULL']}").expanduser().resolve())
print("✅ Environment variables set:")
print("OUT_DIR_FULL=", os.environ["OUT_DIR_FULL"])

✅ Environment variables set:
OUT_DIR_FULL= /home/jovyan/llm_spine_parse_train_public/training_output/qwen2_5_vl_lora_aggplus


# Smoke Test

In [7]:
%%bash

python ../train_qwen_vl_lora_qlora.py \
    --model_id Qwen/Qwen2.5-VL-7B-Instruct \
    --profile nrp_a100_32gb \
    --data_file ../input/512/train.jsonl \
    --eval_data_file ../input/512/eval.jsonl \
    --output_dir "$OUT_DIR_FULL" \
    --num_epochs 1 \
    --max_samples 100 \
    --evaluation_strategy epoch \
    --save_steps 50 \
    --logging_steps 10


QWEN2.5-VL QLORA FINE-TUNING PIPELINE

✓ Using explicit profile: nrp_a100_32gb

📋 Loading profile: nrp_a100_32gb
   NRP A100 32GB - Production training (high quality)

✓ Loaded 100 training examples
✓ Loaded 273 eval examples from ../input/512/eval.jsonl
TRAINING CONFIGURATION SUMMARY
Data file: ../input/512/train.jsonl
Number of examples: 100
Output directory: ~/llm_spine_parse_train_public/training_output/qwen2_5_vl_lora_full
Epochs: 1
Batch size: 2
Gradient accumulation: 16
Effective batch size: 32
Learning rate: 0.0002
LoRA rank: 16
LoRA alpha: 32
Gradient checkpointing: False
Max sequence length: 1024
[1/3] Setting up model configuration...
🔍 Detected CUDA GPU (Tesla V100-SXM2-32GB) with 31.7GB VRAM, using torch.bfloat16
Loading model Qwen/Qwen2.5-VL-7B-Instruct on cuda with dtype=torch.bfloat16, use_4bit=True...


Loading checkpoint shards: 100%|██████████| 5/5 [00:18<00:00,  3.71s/it]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Applying LoRA configuration...
trainable params: 42,543,104 || all params: 8,334,709,760 || trainable%: 0.5104

[2/3] Preparing dataset...
Sample keys: ['pixel_values', 'input_ids', 'attention_mask', 'labels', 'image_grid_thw']
  pixel_values: shape (72, 1176) dtype torch.float32
  input_ids: shape (80,) dtype torch.int64
  attention_mask: shape (80,) dtype torch.int64
  labels: shape (80,) dtype torch.int64
  image_grid_thw: shape (1, 3) dtype torch.int64
[3/3] Setting up trainer...

STARTING TRAINING

Device map: {'': 0}
First trainable parameter names (sample):
   base_model.model.model.visual.blocks.0.mlp.gate_proj.lora_A.default.weight


  0%|          | 0/4 [00:00<?, ?it/s]/opt/conda/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.13/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
100%|█████████▉| 272/273 [03:16<00:00,  1.40it/s]
                                             t/s]
100%|██████████| 4/4 [05:19<00:00, 79.82s/it]    


{'eval_loss': 8.335721015930176, 'eval_runtime': 197.8975, 'eval_samples_per_second': 1.38, 'eval_steps_per_second': 1.38, 'epoch': 1.0}
{'train_runtime': 319.2716, 'train_samples_per_second': 0.313, 'train_steps_per_second': 0.013, 'train_loss': 8.369058609008789, 'epoch': 1.0}

FINAL EVALUATION



100%|██████████| 273/273 [03:16<00:00,  1.39it/s]


Eval metrics: {'eval_loss': 8.335721015930176, 'eval_runtime': 197.4949, 'eval_samples_per_second': 1.382, 'eval_steps_per_second': 1.382, 'epoch': 1.0, 'eval_perplexity': 4170.207258893119}
⚠️  Failed to save eval metrics: [Errno 2] No such file or directory: '~/llm_spine_parse_train_public/training_output/qwen2_5_vl_lora_full/metrics_eval.json'

SAVING ADAPTERS

✓ Saved LoRA adapters to ~/llm_spine_parse_train_public/training_output/qwen2_5_vl_lora_full/adapter_model
✓ Saved tokenizer to ~/llm_spine_parse_train_public/training_output/qwen2_5_vl_lora_full/adapter_model

TRAINING COMPLETE

Next steps:
1. Test adapters: python test_lora_inference.py --adapter_dir ~/llm_spine_parse_train_public/training_output/qwen2_5_vl_lora_full/adapter_model --test_image img/segments/test.jpg
2. Launch vLLM with base + adapter: python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-VL-7B-Instruct --enable-lora --lora-modules spine_adapter=~/llm_spine_parse_train_public/training_output/qwen2

# Full Training

In [5]:
%%bash

nohup env \
  PYTHONUNBUFFERED=1 \
  python ../train_qwen_vl_lora_qlora.py \
    --model_id Qwen/Qwen2.5-VL-7B-Instruct \
    --num_epochs=3 \
    --profile nrp_a100_32gb \
    --data_file ../input/512/train.jsonl \
    --eval_data_file ../input/512/eval.jsonl \
    --output_dir "$OUT_DIR_FULL" \
    --evaluation_strategy epoch \
    --load_best_model_at_end \
    --early_stopping_patience 3 \
    > train_full.log 2>&1 &

echo "Training started in background. Log file: train_full.log"
echo "Check progress with: tail -f train_full.log"

Training started in background. Log file: train_full.log
Check progress with: tail -f train_full.log
